In [2]:


def resolve_path(p):
    p = Path(p)
    if p.is_absolute():
        return p
    return PROJECT_ROOT / p


def get_debug_pitch_path(recording_id, root_dir="data/interim"):
    root_dir = resolve_path(root_dir)
    return root_dir / recording_id / "pitch" / f"{recording_id}_pitch_preprocessed_debug.parquet"



recording_id = S.SARASUDA_VARNAM[0]

debug_path = get_debug_pitch_path(recording_id)

df_debug = pl.read_parquet(debug_path)

print(debug_path)
print(df_debug.shape)
display(df_debug.head().to_pandas())

/home/lluis/master-thesis/CSISD/data/interim/srs_v1_bdn_sav/pitch/srs_v1_bdn_sav_pitch_preprocessed_debug.parquet
(43805, 10)


,time_rel_sec,f0_Hz,f0_pchip,f0_savgol_p3_w13,f0_savgol_p3_w27,f0_interpolated,too_long_to_interp,is_outlier,valid_for_pchip,group_id
0,0.000000,0.0,NaN,NaN,NaN,0.0,True,False,False,NaN
1,0.010001,0.0,NaN,NaN,NaN,0.0,True,False,False,0.0
2,0.020001,0.0,NaN,NaN,NaN,0.0,True,False,False,0.0
3,0.030002,0.0,NaN,NaN,NaN,0.0,True,False,False,0.0
4,0.040003,0.0,NaN,NaN,NaN,0.0,True,False,False,0.0


In [3]:

def hz_to_cents(f_hz, tonic_hz):
    f_hz = np.asarray(f_hz, dtype=float)
    out = np.full_like(f_hz, np.nan, dtype=float)
    mask = np.isfinite(f_hz) & (f_hz > 0)
    out[mask] = 1200.0 * np.log2(f_hz[mask] / tonic_hz)
    return out


def extract_audio_window(audio_path, t_start, window_sec):
    audio_path = Path(audio_path)

    if not audio_path.exists():
        raise FileNotFoundError(f"No existeix l'àudio: {audio_path}")

    y, sr = sf.read(audio_path)

    n_samples = len(y)
    audio_dur = n_samples / sr

    t0 = max(0.0, float(t_start))
    t1 = min(float(t_start + window_sec), audio_dur)

    s0 = int(round(t0 * sr))
    s1 = int(round(t1 * sr))

    y_win = y[s0:s1]
    return y_win, sr, t0, t1, audio_dur


def find_true_runs(mask_bool):
    mask_bool = np.asarray(mask_bool, dtype=bool)
    runs = []

    in_run = False
    start = None

    for i, v in enumerate(mask_bool):
        if v and not in_run:
            start = i
            in_run = True
        elif not v and in_run:
            runs.append((start, i))
            in_run = False

    if in_run:
        runs.append((start, len(mask_bool)))

    return runs


def find_group_runs(group_ids):
    g = np.asarray(group_ids)
    runs = []

    if len(g) == 0:
        return runs

    start = 0
    current = g[0]

    for i in range(1, len(g)):
        if g[i] != current:
            runs.append((start, i, current))
            start = i
            current = g[i]

    runs.append((start, len(g), current))
    return runs

def get_overlay_runs(df: pl.DataFrame, overlay_col: str, time_col: str = "time_rel_sec"):
    """
    Retorna una llista de runs per un overlay booleà:
    [
        {
            "start_idx": s,
            "end_idx": e,              # exclusiu
            "x_left": left,
            "x_right": right,
            "x_mid": mid,
            "n_samples": e - s,
        },
        ...
    ]
    """
    if overlay_col not in df.columns:
        return []

    df_sorted = df.sort(time_col)

    t = df_sorted[time_col].to_numpy()
    mask = df_sorted[overlay_col].fill_null(False).to_numpy().astype(bool)

    runs = find_true_runs(mask)
    out = []

    for s, e in runs:
        run_len = e - s

        if run_len == 1:
            x_mid = float(t[s])
            left = x_mid
            right = x_mid
        else:
            x0 = t[s]
            x1 = t[e - 1]
            left = 0.5 * (t[s - 1] + t[s]) if s > 0 else x0
            right = 0.5 * (t[e - 1] + t[e]) if e < len(t) else x1
            x_mid = 0.5 * (left + right)

        out.append({
            "start_idx": s,
            "end_idx": e,
            "x_left": float(left),
            "x_right": float(right),
            "x_mid": float(x_mid),
            "n_samples": int(run_len),
        })

    return out


In [ ]:
def plot_pitch_range_flexible(
    df: pl.DataFrame,
    t_start: float,
    t_end: float,
    tonic_hz: float,
    time_col: str = "time_rel_sec",
    hz_cols=None,
    overlay_cols=None,
    group_col: str = "group_id",
    figsize=(15, 6),
    show_overlay_counts: bool = False,
):
    if hz_cols is None:
        hz_cols = []
    if overlay_cols is None:
        overlay_cols = []

    df_win = (
        df.filter((pl.col(time_col) >= t_start) & (pl.col(time_col) <= t_end))
          .sort(time_col)
    )

    if df_win.height == 0:
        print("No hi ha dades en aquest rang temporal.")
        return None

    t = df_win[time_col].to_numpy()
    fig, ax = plt.subplots(figsize=figsize)

    # --------------------------------------------------
    # group_id com regions alternes
    # --------------------------------------------------
    if group_col in overlay_cols and group_col in df_win.columns:
        g_series = (
            df_win[group_col]
            .fill_null(strategy="forward")
            .fill_null(strategy="backward")
        )

        g = g_series.to_numpy()
        runs = find_group_runs(g)

        first_group_label = True

        for k, (s, e, gid) in enumerate(runs):
            x0 = t[s]
            x1 = t[e - 1]

            left = 0.5 * (t[s - 1] + t[s]) if s > 0 else x0
            right = 0.5 * (t[e - 1] + t[e]) if e < len(t) else x1

            color = "lightgray" if (k % 2 == 0) else "gainsboro"

            ax.axvspan(
                left,
                right,
                color=color,
                alpha=0.20,
                label="group_id" if first_group_label else None,
                zorder=1,
            )
            first_group_label = False

    # --------------------------------------------------
    # Corbes Hz -> cents amb jerarquia visual
    # --------------------------------------------------
    plot_order = [
        "f0_savgol_p3_w27",
        "f0_savgol_p3_w13",
        "f0_pchip",
        "f0_Hz",
    ]

    styles = {
        "f0_Hz": dict(
            color="#0026ff",
            linewidth=1.2,
            alpha=1,
            zorder=40,
        ),
        "f0_pchip": dict(
            color="#ff0077",
            linewidth=1.2,
            alpha=1,
            zorder=30,
        ),
        "f0_savgol_p3_w13": dict(
            color="#f88b42",
            linewidth=3.5,
            alpha=0.85,
            zorder=10,
        ),
        "f0_savgol_p3_w27": dict(
            color="#9467bd",
            linewidth=3.5,
            alpha=0.75,
            zorder=5,
        ),
    }

    for col in plot_order:
        if col not in hz_cols or col not in df_win.columns:
            continue

        y_hz = df_win[col].to_numpy()
        y_cents = hz_to_cents(y_hz, tonic_hz)

        ax.plot(
            t,
            y_cents,
            label=col,
            **styles[col]
        )

    # Fixem límits Y abans de posar etiquetes
    ymin, ymax = ax.get_ylim()
    yrange = ymax - ymin if np.isfinite(ymax - ymin) and ymax > ymin else 1.0

    # alçades relatives de les etiquetes, sempre cap a dalt
    overlay_label_y = {
        "too_long_to_interp": ymax - 0.06 * yrange,
        "is_outlier": ymax - 0.12 * yrange,
        "valid_for_pchip": ymax - 0.18 * yrange,
    }

    # --------------------------------------------------
    # Overlays booleans
    # --------------------------------------------------
    overlay_styles = {
        "too_long_to_interp": {"color": "orange", "alpha": 0.16, "line_alpha": 0.16},
        "is_outlier": {"color": "blue", "alpha": 0.18, "line_alpha": 0.18},
        "valid_for_pchip": {"color": "green", "alpha": 0.12, "line_alpha": 0.12},
    }

    for bcol in overlay_cols:
        if bcol == group_col:
            continue
        if bcol not in df_win.columns:
            continue

        mask = df_win[bcol].fill_null(False).to_numpy().astype(bool)
        runs = find_true_runs(mask)

        line_labeled = False
        region_labeled = False
        style = overlay_styles.get(bcol, {"color": "black", "alpha": 0.15})
        y_text = overlay_label_y.get(bcol, ymax - 0.1 * yrange)

        for s, e in runs:
            run_len = e - s

            if run_len == 1:
                x_mid = t[s]

                ax.axvline(
                    x=t[s],
                    color=style["color"],
                    linestyle="--",
                    linewidth=1.2,
                    alpha=style.get("line_alpha", style["alpha"]),
                    label=f"{bcol} (dot)" if not line_labeled else None,
                    zorder=20,
                )
                line_labeled = True

            else:
                x0 = t[s]
                x1 = t[e - 1]

                left = 0.5 * (t[s - 1] + t[s]) if s > 0 else x0
                right = 0.5 * (t[e - 1] + t[e]) if e < len(t) else x1
                x_mid = 0.5 * (left + right)

                ax.axvspan(
                    left,
                    right,
                    color=style["color"],
                    alpha=style["alpha"],
                    label=f"{bcol} (region)" if not region_labeled else None,
                    zorder=15,
                )
                region_labeled = True

            if show_overlay_counts:
                ax.text(
                    x_mid,
                    y_text,
                    str(run_len),
                    ha="center",
                    va="center",
                    fontsize=9,
                    color=style["color"],
                    bbox=dict(
                        boxstyle="round,pad=0.18",
                        facecolor="white",
                        edgecolor=style["color"],
                        alpha=0.85,
                    ),
                    zorder=50,
                    clip_on=False,
                )

    ax.set_xlabel(time_col)
    ax.set_ylabel("cents")
    ax.set_title(f"Pitch between {t_start:.2f}s and {t_end:.2f}s")
    ax.grid(True, alpha=0.25)
    ax.legend(loc="best")
    plt.tight_layout()
    plt.show()

    return df_win

In [ ]:
def interactive_pitch_audio_viewer_flexible(
    df: pl.DataFrame,
    audio_path,
    tonic_hz: float,
    time_col: str = "time_rel_sec",
    default_window_sec: float = 5.0,
    default_start_sec: float | None = None,
    step_sec: float = 0.25,
):
    if df.height == 0:
        print("DataFrame buit.")
        return

    if time_col not in df.columns:
        raise ValueError(f"Column {time_col} not found in DataFrame")

    hz_candidates = [
        c for c in [
            "f0_Hz",
            "f0_pchip",
            "f0_savgol_p3_w13",
            "f0_savgol_p3_w27",
        ]
        if c in df.columns
    ]

    overlay_candidates = [
        c for c in [
            "too_long_to_interp",
            "is_outlier",
            "valid_for_pchip",
            "group_id",
        ]
        if c in df.columns
    ]

    t_min = float(df[time_col].min())
    t_max = float(df[time_col].max())

    audio_info = sf.info(str(audio_path))
    audio_dur = float(audio_info.frames) / float(audio_info.samplerate)

    global_min = max(0.0, t_min)
    global_max = min(t_max, audio_dur)

    if global_max <= global_min:
        print("There is no overlapping time range between pitch data and audio.")
        return

    if default_start_sec is None:
        default_start_sec = global_min

    navigable_overlay_candidates = [
        c for c in [
            "too_long_to_interp",
            "is_outlier",
            "valid_for_pchip",
        ]
        if c in df.columns
    ]

    overlay_runs_map = {
        col: get_overlay_runs(df, col, time_col=time_col)
        for col in navigable_overlay_candidates
    }    

    default_window_sec = min(default_window_sec, global_max - global_min)
    default_start_sec = min(
        max(default_start_sec, global_min),
        max(global_min, global_max - default_window_sec)
    )

    window_slider = widgets.FloatSlider(
        value=default_window_sec,
        min=0.5,
        max=max(0.5, min(30.0, global_max - global_min)),
        step=0.25,
        description="finestra (s)",
        continuous_update=False,
        readout_format=".2f",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="650px"),
    )

    start_slider = widgets.FloatSlider(
        value=default_start_sec,
        min=global_min,
        max=max(global_min, global_max - default_window_sec),
        step=step_sec,
        description="inici (s)",
        continuous_update=False,
        readout_format=".2f",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="650px"),
    )

    prev_button = widgets.Button(
        description="← finestra anterior",
        layout=widgets.Layout(width="180px"),
    )

    next_button = widgets.Button(
        description="següent finestra →",
        layout=widgets.Layout(width="180px"),
    )

    hz_select = widgets.SelectMultiple(
        options=hz_candidates,
        value=tuple([c for c in ["f0_Hz", "f0_savgol_p3_w13"] if c in hz_candidates]),
        description="corbes",
        rows=min(4, max(2, len(hz_candidates))),
        style={"description_width": "initial"},
        layout=widgets.Layout(width="260px", height="110px"),
    )

    overlay_select = widgets.SelectMultiple(
        options=overlay_candidates,
        value=tuple([c for c in ["too_long_to_interp", "is_outlier", "valid_for_pchip"] if c in overlay_candidates]),
        description="overlays",
        rows=min(4, max(2, len(overlay_candidates))),
        style={"description_width": "initial"},
        layout=widgets.Layout(width="260px", height="110px"),
    )

    overlay_nav_dropdown = widgets.Dropdown(
        options=navigable_overlay_candidates,
        value=navigable_overlay_candidates[0] if len(navigable_overlay_candidates) > 0 else None,
        description="navegar overlay",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="260px"),
    )

    prev_overlay_button = widgets.Button(
        description="← overlay anterior",
        layout=widgets.Layout(width="180px"),
    )

    next_overlay_button = widgets.Button(
        description="overlay següent →",
        layout=widgets.Layout(width="180px"),
    )

    autoplay_checkbox = widgets.Checkbox(
        value=False,
        description="autoplay àudio",
        indent=False,
    )

    overlay_counts_checkbox = widgets.Checkbox(
        value=False,
        description="mostrar n samples overlays",
        indent=False,
    )

    info_box = widgets.Output()
    plot_box = widgets.Output()
    audio_box = widgets.Output()

    def _sync_start_bounds(*args):
        w = float(window_slider.value)
        start_slider.max = max(global_min, global_max - w)
        if start_slider.value > start_slider.max:
            start_slider.value = start_slider.max

    def _go_prev(_):
        w = float(window_slider.value)
        new_start = max(global_min, float(start_slider.value) - w)
        start_slider.value = new_start

    def _go_next(_):
        w = float(window_slider.value)
        new_start = min(start_slider.max, float(start_slider.value) + w)
        start_slider.value = new_start

    prev_button.on_click(_go_prev)
    next_button.on_click(_go_next)

    window_slider.observe(_sync_start_bounds, names="value")

    def _go_prev_overlay(_):
        _jump_to_overlay_run(direction=-1)

    def _go_next_overlay(_):
        _jump_to_overlay_run(direction=1)

    prev_overlay_button.on_click(_go_prev_overlay)
    next_overlay_button.on_click(_go_next_overlay)

    def _refresh(*args):
        t0 = float(start_slider.value)
        w = float(window_slider.value)
        t1 = min(t0 + w, global_max)

        selected_hz = list(hz_select.value)
        selected_overlays = list(overlay_select.value)

        info_box.clear_output(wait=True)
        plot_box.clear_output(wait=True)
        audio_box.clear_output(wait=True)

        with info_box:
            print(f"Finestra visible: {t0:.2f}s → {t1:.2f}s")
            print(f"Pitch disponible: {t_min:.2f}s → {t_max:.2f}s")
            print(f"Àudio disponible: 0.00s → {audio_dur:.2f}s")
            print(f"Corbes visibles: {selected_hz if selected_hz else 'cap'}")
            print(f"Overlays visibles: {selected_overlays if selected_overlays else 'cap'}")
            nav_overlay = overlay_nav_dropdown.value
            if nav_overlay is not None:
                n_runs = len(overlay_runs_map.get(nav_overlay, []))
                print(f"Runs disponibles per {nav_overlay}: {n_runs}")

        with plot_box:
            plot_pitch_range_flexible(
                df=df,
                t_start=t0,
                t_end=t1,
                tonic_hz=tonic_hz,
                time_col=time_col,
                hz_cols=selected_hz,
                overlay_cols=selected_overlays,
                group_col="group_id",
                figsize=(15, 6),
                show_overlay_counts=overlay_counts_checkbox.value,
            )

        y_win, sr, a0, a1, _ = extract_audio_window(
            audio_path=audio_path,
            t_start=t0,
            window_sec=(t1 - t0),
        )

        with audio_box:
            print(f"Àudio reproduït: {a0:.2f}s → {a1:.2f}s")
            display(
                Audio(
                    data=y_win.T if y_win.ndim == 2 else y_win,
                    rate=sr,
                    autoplay=autoplay_checkbox.value,
                )
            )

    def _center_window_on_x(x_mid):
        w = float(window_slider.value)
        new_start = x_mid - 0.5 * w
        new_start = max(global_min, min(new_start, start_slider.max))
        start_slider.value = new_start

    def _jump_to_overlay_run(direction=1):
        overlay_name = overlay_nav_dropdown.value
        if overlay_name is None:
            return

        runs = overlay_runs_map.get(overlay_name, [])
        if len(runs) == 0:
            return

        current_t0 = float(start_slider.value)
        current_t1 = min(current_t0 + float(window_slider.value), global_max)
        current_center = 0.5 * (current_t0 + current_t1)

        mids = np.array([r["x_mid"] for r in runs], dtype=float)

        if direction > 0:
            # primer run estrictament després del centre actual
            idxs = np.where(mids > current_center + 1e-12)[0]
            idx = int(idxs[0]) if len(idxs) > 0 else len(runs) - 1
        else:
            # últim run estrictament abans del centre actual
            idxs = np.where(mids < current_center - 1e-12)[0]
            idx = int(idxs[-1]) if len(idxs) > 0 else 0

        _center_window_on_x(runs[idx]["x_mid"])
    
    start_slider.observe(_refresh, names="value")
    window_slider.observe(_refresh, names="value")
    hz_select.observe(_refresh, names="value")
    overlay_select.observe(_refresh, names="value")
    autoplay_checkbox.observe(_refresh, names="value")
    overlay_counts_checkbox.observe(_refresh, names="value")

    controls = widgets.VBox([
        window_slider,
        start_slider,
        widgets.HBox([prev_button, next_button]),
        widgets.HBox([hz_select, overlay_select]),
        widgets.HBox([overlay_nav_dropdown, prev_overlay_button, next_overlay_button]),
        widgets.HBox([autoplay_checkbox, overlay_counts_checkbox]),
    ])
        

    display(controls, info_box, plot_box, audio_box)

    _sync_start_bounds()
    _refresh()

In [9]:
AUDIO_PATH = S.DATA_CORPUS / recording_id / "audio" / f"{recording_id}.mp3"
TONIC_HZ = S.SARASUDA_TONICS[recording_id]


interactive_pitch_audio_viewer_flexible(
    df=df_debug,
    audio_path=AUDIO_PATH,
    tonic_hz=TONIC_HZ,
    time_col="time_rel_sec",
    default_window_sec=5.0,
    default_start_sec=0.0,
    step_sec=0.25,
)

Output()

Output()

Output()